# 31. Domain Lexicon v1 → v1.1 — 측정 결과 반영

| | |
|---|---|
| 만드는 것 | `data/scent_knowledge/domain_lexicon_v1_1.csv` |
| 근거 | 노트북 30의 `ai_summary` 배수 측정 · `DECISIONS.md` N4 |
| API | **호출하지 않는다** |
| 작성 | 2026-09-11 |

## 적용하는 판정 기준 4가지 — 사용자 승인 2026-09-11

| # | 질문 | 결정 |
|---|---|---|
| 1 | 팀 문서와 데이터가 엇갈리면 | **역할 분리.** 팀 문서는 *"이 표현이 향 표현이다"*, `ai_summary` 는 *"어느 accord 인가"* 를 담당한다 |
| 2 | 배수 문턱 | **1.4.** 통제군(무작위 추출)의 최대 배수가 1.38이므로 그 이하는 잡음과 구별되지 않는다 |
| 3 | 문턱 미달이면 | **`core` → `optional` 로 내리고 근거 칸에 기록한다.** 행을 지우거나 다른 accord 로 교체하지 않는다 |
| 4 | `core` 가 2개 미만이 되면 | **`NO_MAPPING` 으로 돌린다.** spec §4.3 의 단독 매핑 금지 규칙을 충족하지 못하므로 |

### 3번을 교체가 아니라 강등으로 정한 이유

데이터가 가리키는 accord 로 **교체하지 않는다.** `포근한` 의 경우 배수 1위가 `savory`(3.72x)지만
방향을 뒤집어 보면 이야기가 다르다.

| accord | 보유 향수 | cozy 향수 중 비율 | **그 accord 중 cozy 비율** |
|---|---:|---:|---:|
| `savory` | 115 | 3.4% | **82.6%** |
| `powdery` | 6,291 | **68.2%** | 29.9% |

우리 서비스는 **accord 로 향수를 골라 주는** 쪽이므로 오른쪽 열이 중요하다.
`savory` 는 맞히면 잘 맞지만 줄 수 있는 향수가 115개뿐이고, 조합하면
`savory+nutty+cacao` 가 13만 개 중 **5개**다. 측정 기록 1번의 `foresty` 문제와 같다.

**`powdery` 는 틀린 것이 아니라 필수로 걸기엔 너무 넓다.** 그것은 `optional` 로 내리면
해결되는 문제이지 다른 accord 로 갈아끼울 문제가 아니다.

## 0. 실행 조건과 한계

### 노트북 30의 판정을 그대로 쓰지 않았다

노트북 30은 `min_support = 30` 을 걸었다. 해당군에서 30개 미만인 accord 는 버린다는 규칙이다.
그런데 **희귀 accord 가 작은 집단에서 30개를 못 넘는 것은 당연하므로, 이 규칙은 희귀
accord 를 구조적으로 차별한다.**

`반증 — 해당군에 지지도 없음` 3건을 원자료로 다시 셌다.

| 표현 | accord | 해당군 | 보유 | 배수 | 재판정 |
|---|---:|---:|---:|---:|---|
| 머스크 | `soapy` | 653 | 14 | **2.15x** | **문턱 통과** — 30개 미만이라 표에서 빠졌을 뿐 |
| 이불 | `soapy` | 112 | 2 | 1.79x | **표본 부족** — 2개로 낸 배수는 의미 없음 |
| 호텔 | `soapy` | 1,163 | 15 | 1.29x | 문턱 미달 (판정 유지) |

`머스크 → soapy` 는 기계적으로 적용했다면 **잘못 강등할 뻔했다.**
이 매핑은 `nlr_engineering_notes.md` 2번이 코퍼스로 확인한 한국어 드리프트 항목이다.

### 한계

- **`status` 는 바뀌지 않는다.** 매핑 행은 여전히 전부 `candidate` 다.
  강등은 검색 동작을 안전하게 바꾼 것이지 의미 판정이 끝났다는 뜻이 아니다
- **근거가 `ai_summary` 하나다.** Fragrantica 의 AI 가 사용자 리뷰를 요약한 것이고,
  커버리지가 9.4% 이며 인기 향수에 치우쳐 있다. `DECISIONS.md` N4 의 한계가 그대로 적용된다
- **`NO_MAPPING` 으로 돌린 3개는 되돌릴 수 있다.** 행을 지우지 않았고 근거도 남겼다
- **`spec.md` §3 의 워크드 예제가 `포근한 → powdery` 다.** 이 노트북은 사전만 바꾸며
  spec 본문은 고치지 않는다. 문서 수정은 별도 판단이다

### 하지 않는 것

`domain_lexicon_v1.csv` 원본 수정(새 파일로 만든다), 행 삭제, accord 교체,
`spec.md` 수정, 평가 데이터 수정, LLM 호출.

In [1]:
import hashlib
import json
import pathlib

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", 240)

# True면 계산과 표시만 하고 파일을 만들지 않는다.
REPORT_ONLY = False

LIFT_THRESHOLD = 1.4     # 통제군 최대 1.38
MIN_CORE = 2             # spec §4.3 단독 매핑 금지
STAMP = "[30] "          # 근거 칸에 붙일 출처 표시

print("REPORT_ONLY:", REPORT_ONLY, "/ 문턱:", LIFT_THRESHOLD)

REPORT_ONLY: False / 문턱: 1.4


## 1. 경로 · 입력 해싱 · 쓰기 가드

**`domain_lexicon_v1.csv` 를 보호 목록에 넣는다.** 원본을 덮어쓰지 않고 v1.1 을 새로 만든다.

In [2]:
PROJECT_ROOT = pathlib.Path.cwd()
KNOWLEDGE_DIR = PROJECT_ROOT / "data" / "scent_knowledge"
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"

INPUT_PATHS = {
    "lexicon_v1": KNOWLEDGE_DIR / "domain_lexicon_v1.csv",
    "crosscheck": OUTPUT_DIR / "30_lexicon_crosscheck.csv",
    "lift": OUTPUT_DIR / "30_expression_accord_lift.csv",
}
OUTPUT_PATHS = {
    "lexicon_v11": KNOWLEDGE_DIR / "domain_lexicon_v1_1.csv",
    "changelog": OUTPUT_DIR / "31_lexicon_v1_1_changelog.md",
}


def sha256_file(path):
    """파일의 SHA-256 hex digest. str."""
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


missing = [str(p) for p in INPUT_PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")
input_hashes_before = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}

PROTECTED = {p.resolve() for p in INPUT_PATHS.values()} | {
    (KNOWLEDGE_DIR / "community_product_alias_v1.csv").resolve(),
    (PROJECT_ROOT / "evaluation_data" / "stage1"
     / "13_stage1_golden_set_v1_200.xlsx").resolve(),
    (PROJECT_ROOT / "perfumes.jsonl").resolve(),
    (PROJECT_ROOT / "perfumes.csv").resolve(),
}
ALLOWED_WRITES = {p.resolve() for p in OUTPUT_PATHS.values()}


def write_output(path, writer):
    """OUTPUT_PATHS 의 경로에만 쓴다. REPORT_ONLY면 생략. pathlib.Path 또는 None."""
    path = pathlib.Path(path).resolve()
    if path not in ALLOWED_WRITES:
        raise RuntimeError(f"쓰기 허용 경로가 아닙니다: {path}")
    if path in PROTECTED:
        raise RuntimeError(f"보호 파일 덮어쓰기 시도: {path.name}")
    if REPORT_ONLY:
        print(f"[REPORT_ONLY] 저장 생략: {path.name}")
        return None
    writer(path)
    print(f"저장: {path.relative_to(PROJECT_ROOT)}")
    return path


display(pd.Series(input_hashes_before, name="sha256").str.slice(0, 16).to_frame())

,sha256
lexicon_v1,936808a9b3c32c2b
crosscheck,41865413e557214f
lift,ea961c066537e78f


## 2. 사전 등록 — 행별 조치를 데이터 로드 전에 고정한다

각 (표현, accord) 에 어떤 조치를 할지, 그 근거 문구가 무엇인지 미리 적는다.
`keep` 은 유지, `demote` 는 `core` → `optional`, `sample` 은 표본 부족으로 판정하지 않음이다.

In [3]:
DECISION = {
    ("머스크", "musky"):        ("keep",   "2.19x·1위",  "검색어가 accord 이름과 같아 순환이나, 이름 동일성 매핑이므로 배수 근거가 필요 없다"),
    ("머스크", "soapy"):        ("keep",   "2.15x",      "해당군 653개 중 14개. 노트북 30의 최소 지지도 30에 걸려 표에서 빠졌으나 배수는 문턱을 넘는다"),
    ("머스크", "fresh"):        ("keep",   "1.27x·8위",  "문턱 1.4 미달이나 원래 optional 이므로 조치 없음"),
    ("포근한", "powdery"):      ("demote", "1.35x·14위", "통제군 최대 1.38 보다 낮다. 다만 cozy 향수 2,756개 중 68.2%가 보유하므로 틀린 것이 아니라 필수로 걸기엔 너무 넓다"),
    ("포근한", "musky"):        ("demote", "1.12x·21위", "통제군 수준. cozy 향수 중 보유 35.8%"),
    ("포근한", "vanilla"):      ("keep",   "1.56x·10위", "유일하게 문턱을 넘었다. cozy 향수 중 55.1% 보유"),
    ("이불",   "powdery"):      ("keep",   "1.57x·2위",  ""),
    ("이불",   "soapy"):        ("sample", "해당군 112개 중 2개", "2개로 계산한 배수는 의미가 없다. 지지도 반증도 아니다"),
    ("이불",   "musky"):        ("keep",   "1.45x",      "재계산 결과 문턱을 넘는다. 원래 optional 이므로 조치 없음"),
    ("차가운", "ozonic"):       ("demote", "0.83x·41위", "전체보다 오히려 드물다"),
    ("차가운", "fresh"):        ("demote", "0.91x·33위", "전체보다 오히려 드물다"),
    ("호텔",   "soapy"):        ("demote", "1.29x",      "해당군 1,163개 중 15개. 문턱 미달"),
    ("호텔",   "white floral"): ("demote", "1.10x·14위", "문턱 미달"),
}
NOTE_ONLY = {  # 지지된 행에 배수를 근거로 덧붙이기만 한다
    ("깨끗한", "soapy"): "2.18x·1위", ("깨끗한", "fresh"): "1.61x·5위",
    ("깨끗한", "aquatic"): "1.77x·2위",
    ("빨래", "soapy"): "4.50x·1위", ("빨래", "fresh"): "1.77x·5위(순환)",
    ("비 오는 숲", "mossy"): "3.63x·1위", ("비 오는 숲", "earthy"): "3.04x·3위",
    ("비 오는 숲", "green"): "2.00x·5위",
    ("촉촉한", "aquatic"): "3.30x·1위", ("촉촉한", "green"): "2.12x·4위",
    ("휴양지", "tropical"): "5.74x·2위(순환)", ("휴양지", "coconut"): "5.76x·1위",
    ("달달", "sweet"): "1.26x·16위(순환)", ("달달", "caramel"): "1.61x·1위",
}

print(f"조치 지정 {len(DECISION)}행 / 지지 기록만 {len(NOTE_ONLY)}행")
for action in ("keep", "demote", "sample"):
    n = sum(1 for a, _, _ in DECISION.values() if a == action)
    print(f"  {action:8s} {n}행")

조치 지정 13행 / 지지 기록만 14행
  keep     6행
  demote   6행
  sample   1행


## 3. 반영

In [4]:
lex = pd.read_csv(INPUT_PATHS["lexicon_v1"], keep_default_na=False, dtype=str)
before = lex.copy()

changed = demoted = noted = 0
for i, r in lex.iterrows():
    if r["candidate_type"] != "ACCORD":
        continue
    key = (r["expression"], r["candidate_name"])
    if key in DECISION:
        action, lift, why = DECISION[key]
        add_text = f"{STAMP}ai_summary 배수 {lift}."
        if why:
            add_text += f" {why}"
        if action == "demote":
            lex.at[i, "required"] = "optional"
            add_text += " → 필수에서 보조로 내렸다."
            demoted += 1
        elif action == "sample":
            add_text += " → 판정하지 않는다."
        lex.at[i, "rationale"] = r["rationale"] + " " + add_text
        changed += 1
    elif key in NOTE_ONLY:
        lex.at[i, "rationale"] = (r["rationale"]
                                  + f" {STAMP}ai_summary 배수 {NOTE_ONLY[key]}로 지지됨.")
        noted += 1

print(f"근거 갱신 {changed}행 (강등 {demoted}행) / 지지 기록만 {noted}행")

근거 갱신 13행 (강등 6행) / 지지 기록만 15행


### `core` 가 2개 미만이 된 갈래를 `NO_MAPPING` 으로 돌린다

`spec.md` §4.3 의 단독 매핑 금지 규칙은 측정으로 정해진 것이다 —
`citrus` 단독은 후보 59,969개 중 17,736개가 강도 100으로 동점이라 순위가 나오지 않는다
(`DECISIONS.md` N2). 따라서 `core` 가 1개여도 검색이 성립하지 않는다.

In [5]:
to_nomap = []
acc_rows = lex[lex.candidate_type == "ACCORD"]
for expr in sorted(set(acc_rows.expression)):
    rows = acc_rows[acc_rows.expression == expr]
    for cond in sorted(set(rows.match_condition)):
        branch = rows[rows.match_condition == cond]
        n_core = int((branch.required == "core").sum())
        if n_core < MIN_CORE:
            to_nomap.append((expr, cond, n_core))

print(f"core {MIN_CORE}개 미만이 된 갈래 {len(to_nomap)}개")
for expr, cond, n_core in to_nomap:
    print(f"  {expr:10s} core {n_core}개   조건 '{cond[:44] or '(기본)'}'")
    mask = (lex.expression == expr) & (lex.match_condition == cond)
    lex.loc[mask, "target_field"] = "NO_MAPPING"
    lex.loc[mask, "rationale"] = lex.loc[mask, "rationale"] + (
        f" {STAMP}필수 후보가 {n_core}개가 되어 spec §4.3 의 '단독 매핑 금지' 규칙을 "
        "충족하지 못한다. 근거가 더 생길 때까지 NO_MAPPING 으로 둔다.")

core 2개 미만이 된 갈래 3개
  차가운        core 0개   조건 '(기본)'
  포근한        core 0개   조건 '(기본)'
  호텔         core 0개   조건 '(기본)'


## 4. 검증

바꾸지 말아야 할 것이 그대로인지 확인한다.

In [6]:
problems = []
if len(lex) != len(before):
    problems.append(f"행 수가 바뀌었다: {len(before)} → {len(lex)}")
if list(lex.columns) != list(before.columns):
    problems.append("컬럼이 바뀌었다")
for col in ["entry_id", "expression", "aliases", "candidate_type", "candidate_name",
            "rank", "evidence_tier", "corpus_support", "standardness", "source_query_ids"]:
    diff = int((lex[col] != before[col]).sum())
    if diff:
        problems.append(f"바꾸면 안 되는 컬럼이 바뀌었다: {col} {diff}행")
if int((lex.status != "candidate").sum()) != int((before.status != "candidate").sum()):
    problems.append("status 분포가 바뀌었다")
changed_required = int((lex.required != before.required).sum())
if changed_required != demoted:
    problems.append(f"required 변경 수 불일치: {changed_required} != {demoted}")
# 강등된 행이 전부 core → optional 인지
for i in lex.index:
    if lex.at[i, "required"] != before.at[i, "required"]:
        if not (before.at[i, "required"] == "core" and lex.at[i, "required"] == "optional"):
            problems.append(f"core→optional 이 아닌 변경: {lex.at[i, 'entry_id']}")
# 모든 rationale 이 비어 있지 않은지
if (lex.rationale.str.strip() == "").any():
    problems.append("rationale 이 빈 행이 있다")

if problems:
    for p in problems:
        print(" ", p)
    raise RuntimeError(f"검증 실패 — {len(problems)}건")
print("검증 통과")
print(f"  행 {len(lex)} / 컬럼 {len(lex.columns)} — 변경 없음")
print(f"  required 변경 {changed_required}행 — 전부 core → optional")
print(f"  entry_id·expression·candidate_name·evidence_tier·corpus_support 변경 0행")

검증 통과
  행 44 / 컬럼 16 — 변경 없음
  required 변경 6행 — 전부 core → optional
  entry_id·expression·candidate_name·evidence_tier·corpus_support 변경 0행


### 변경 요약

In [7]:
summary = []
for i in lex.index:
    if (lex.at[i, "required"] != before.at[i, "required"]
            or lex.at[i, "target_field"] != before.at[i, "target_field"]):
        summary.append({
            "entry_id": lex.at[i, "entry_id"],
            "표현": lex.at[i, "expression"],
            "accord": lex.at[i, "candidate_name"],
            "required": f"{before.at[i, 'required']} → {lex.at[i, 'required']}"
                        if lex.at[i, "required"] != before.at[i, "required"] else "—",
            "target_field": f"{before.at[i, 'target_field']} → {lex.at[i, 'target_field']}"
                            if lex.at[i, "target_field"] != before.at[i, "target_field"] else "—",
        })
change_df = pd.DataFrame(summary)
display(change_df)

print("\ntarget_field 별 표현 수")
print(lex.groupby("target_field").expression.nunique().to_string())
print("\nACCORD 행의 required")
print(lex[lex.candidate_type == "ACCORD"].required.value_counts().to_string())

,entry_id,표현,accord,required,target_field
0,kr.sens.cozy,포근한,powdery,core → optional,STAGE2_BRIDGE → NO_MAPPING
1,kr.sens.cozy,포근한,musky,core → optional,STAGE2_BRIDGE → NO_MAPPING
2,kr.sens.cozy,포근한,vanilla,—,STAGE2_BRIDGE → NO_MAPPING
3,kr.sens.cold,차가운,ozonic,core → optional,STAGE2_BRIDGE → NO_MAPPING
4,kr.sens.cold,차가운,fresh,core → optional,STAGE2_BRIDGE → NO_MAPPING
5,kr.scene.hotel,호텔,soapy,core → optional,STAGE2_BRIDGE → NO_MAPPING
6,kr.scene.hotel,호텔,white floral,core → optional,STAGE2_BRIDGE → NO_MAPPING



target_field 별 표현 수
target_field
NO_MAPPING       11
STAGE1_DIRECT     7
STAGE2_BRIDGE     8

ACCORD 행의 required
required
core        18
optional    10


## 5. 저장

In [8]:
nomap_lines = "\n".join(
    f"| {expr} | {n_core} | `{cond[:50] or '(기본)'}` |" for expr, cond, n_core in to_nomap)
change_lines = "\n".join(
    f"| `{r['entry_id']}` | {r['표현']} | `{r['accord']}` | {r['required']} | {r['target_field']} |"
    for r in change_df.to_dict("records"))

changelog = f"""# Domain Lexicon v1 → v1.1 변경 기록

`ai_summary` 배수 측정(노트북 30)을 사전에 반영했다. 근거와 한계는 `DECISIONS.md` N4,
측정은 `nlr_engineering_notes.md` 10번.

## 적용한 판정 기준 — 사용자 승인 2026-09-11

| # | 질문 | 결정 |
|---|---|---|
| 1 | 팀 문서와 데이터가 엇갈리면 | **역할 분리.** 팀 문서는 표현의 존재, `ai_summary` 는 accord 지정 |
| 2 | 배수 문턱 | **{LIFT_THRESHOLD}.** 통제군 최대 배수가 1.38이므로 그 이하는 잡음과 구별되지 않는다 |
| 3 | 문턱 미달이면 | **`core` → `optional`, 근거 칸에 기록.** 삭제·교체하지 않는다 |
| 4 | `core` 가 {MIN_CORE}개 미만이면 | **`NO_MAPPING` 으로 돌린다** |

## 노트북 30의 판정을 그대로 쓰지 않은 부분

노트북 30의 `min_support = 30` 은 **희귀 accord 를 구조적으로 차별한다.**
`반증 — 해당군에 지지도 없음` 3건을 원자료로 다시 셌다.

| 표현 | accord | 해당군 | 보유 | 배수 | 재판정 |
|---|---:|---:|---:|---:|---|
| 머스크 | `soapy` | 653 | 14 | **2.15x** | **문턱 통과** — 30개 미만이라 표에서 빠졌을 뿐 |
| 이불 | `soapy` | 112 | 2 | 1.79x | **표본 부족** — 2개로 낸 배수는 의미 없음 |
| 호텔 | `soapy` | 1,163 | 15 | 1.29x | 문턱 미달 (판정 유지) |

기계적으로 적용했다면 `머스크 → soapy` 를 잘못 강등할 뻔했다.
이 매핑은 `nlr_engineering_notes.md` 2번이 코퍼스로 확인한 한국어 드리프트 항목이다.

## 바뀐 행 {len(change_df)}개

| entry_id | 표현 | accord | required | target_field |
|---|---|---|---|---|
{change_lines}

## `NO_MAPPING` 으로 돌린 갈래

| 표현 | 남은 core | 조건 |
|---|---:|---|
{nomap_lines}

`spec.md` §4.3 의 단독 매핑 금지 규칙은 측정으로 정해진 것이다 — `citrus` 단독은 후보
59,969개 중 17,736개가 강도 100으로 동점이라 순위가 나오지 않는다(`DECISIONS.md` N2).
따라서 `core` 가 1개여도 검색이 성립하지 않는다.

## 바뀌지 않은 것

- **행 {len(lex)}개, 컬럼 {len(lex.columns)}개 그대로.** 행을 지우지 않았다
- **accord 를 교체하지 않았다.** 데이터가 가리키는 쪽으로 갈아끼우지 않았다
- `entry_id` · `expression` · `candidate_name` · `evidence_tier` · `corpus_support` ·
  `source_query_ids` **변경 0행**
- **`status` 는 전부 `candidate` 그대로.** 강등은 검색 동작을 안전하게 바꾼 것이지
  의미 판정이 끝났다는 뜻이 아니다
- `domain_lexicon_v1.csv` **원본을 덮어쓰지 않았다**

## 왜 교체가 아니라 강등인가

`포근한` 의 배수 1위는 `savory`(3.72x)다. 그런데 방향을 뒤집으면 다르다.

| accord | 보유 향수 | cozy 향수 중 비율 | 그 accord 중 cozy 비율 |
|---|---:|---:|---:|
| `savory` | 115 | 3.4% | **82.6%** |
| `powdery` | 6,291 | **68.2%** | 29.9% |

서비스는 **accord 로 향수를 골라 주는** 쪽이므로 오른쪽 열이 중요하다. `savory` 는 맞히면
잘 맞지만 줄 향수가 115개뿐이고, `savory+nutty+cacao` 조합은 13만 개 중 **5개**다
(측정 기록 1번의 `foresty` 문제와 같다).

**`powdery` 는 틀린 것이 아니라 필수로 걸기엔 너무 넓다.** `optional` 로 내리면 해결된다.

`savory` 는 넣지 않되 후보로 기록해 둔다 — 그 accord 를 가진 향수의 **82.6%** 가
cozy 로 불리므로 신호는 진짜이나 보유 향수가 115개뿐이다.

## 남은 문제

- **`spec.md` §3 의 워크드 예제가 `포근한 → powdery` 다.** 이 개정으로 사전에서는
  `NO_MAPPING` 이 됐으므로 문서 예제와 데이터가 어긋난다. spec 본문 수정은 별도 판단이다
- **근거가 `ai_summary` 하나다.** 커버리지 9.4%, 인기 향수 편향, Fragrantica 출처 문제가
  그대로 적용된다 (`DECISIONS.md` N4 의 Trade-off)
- **의미 판정은 여전히 없다.** 매핑 행이 전부 `candidate` 인 이유다
"""

write_output(OUTPUT_PATHS["lexicon_v11"],
             lambda p: lex.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["changelog"],
             lambda p: p.write_text(changelog, encoding="utf-8"))

if not REPORT_ONLY:
    reread = pd.read_csv(OUTPUT_PATHS["lexicon_v11"], keep_default_na=False, dtype=str)
    if len(reread) != len(lex) or list(reread.columns) != list(lex.columns):
        raise ValueError("저장 결과 불일치")
    print(f"검증 통과 — {len(reread)}행 / 컬럼 {len(reread.columns)}개")

저장: data\scent_knowledge\domain_lexicon_v1_1.csv
저장: analysis_outputs\31_lexicon_v1_1_changelog.md
검증 통과 — 44행 / 컬럼 16개


## 6. 가드 검증

In [9]:
input_hashes_after = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
changed_inputs = [k for k in input_hashes_before
                  if input_hashes_before[k] != input_hashes_after[k]]
if changed_inputs:
    raise RuntimeError(f"입력이 변경됐습니다: {changed_inputs}")
print("입력 해시 동일 (domain_lexicon_v1.csv 원본 포함)")
for path in sorted(PROTECTED):
    print(f"보호 대상 미변경 확인: {path.name}  {'존재' if path.is_file() else '없음'}")
print("생성한 출력:")
for label, path in OUTPUT_PATHS.items():
    mark = "" if path.is_file() else "  (REPORT_ONLY로 미생성)"
    print(f"  {label}: {path.relative_to(PROJECT_ROOT)}{mark}")

입력 해시 동일 (domain_lexicon_v1.csv 원본 포함)
보호 대상 미변경 확인: 30_expression_accord_lift.csv  존재
보호 대상 미변경 확인: 30_lexicon_crosscheck.csv  존재
보호 대상 미변경 확인: community_product_alias_v1.csv  존재
보호 대상 미변경 확인: domain_lexicon_v1.csv  존재
보호 대상 미변경 확인: 13_stage1_golden_set_v1_200.xlsx  존재
보호 대상 미변경 확인: perfumes.csv  존재
보호 대상 미변경 확인: perfumes.jsonl  존재
생성한 출력:
  lexicon_v11: data\scent_knowledge\domain_lexicon_v1_1.csv
  changelog: analysis_outputs\31_lexicon_v1_1_changelog.md
